In [1]:
pip install torch transformers accelerate matplotlib ipykernel

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch, transformers
print(torch.__version__, transformers.__version__, torch.backends.mps.is_available())

2.6.0+cu124 5.19.0 False


In [1]:
import platform, subprocess, os
import torch, transformers

def comando(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True).stdout.strip()
    except Exception:
        return ""

def obter_distro_linux():
    try:
        with open("/etc/os-release") as f:
            for linha in f:
                if linha.startswith("PRETTY_NAME="):
                    return linha.split("=")[1].strip().strip('"')
    except Exception:
        pass
    return platform.system()

def obter_modelo_cpu():
    try:
        with open("/proc/cpuinfo") as f:
            for linha in f:
                if "model name" in linha:
                    return linha.split(":")[1].strip()
    except Exception:
        pass
    return comando(["lscpu"]).split("\n")[0]

def obter_memoria_total_gb():
    try:
        with open("/proc/meminfo") as f:
            for linha in f:
                if "MemTotal" in linha:
                    kb = int(linha.split()[1])
                    return kb / (1024**2)
    except Exception:
        pass
    return 0.0

# ---- Versões ----
print("=== Software ===")
print(f"Python:        {platform.python_version()} ({platform.machine()})")
print(f"PyTorch:       {torch.__version__}")
print(f"Transformers:  {transformers.__version__}")
print(f"Linux:         {obter_distro_linux()} (Kernel {platform.release()})")

# ---- Chip e memória ----
print("\n=== Hardware ===")
print(f"Processador:   {obter_modelo_cpu()}")
print(f"Núcleos CPU:   {os.cpu_count()}")
print(f"Memória RAM:   {obter_memoria_total_gb():.1f} GB")

# ---- GPU (informações do Linux via nvidia-smi) ----
print("\n=== GPU (Sistema) ===")
gpu_nome = comando(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"])
driver_ver = comando(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"])

if gpu_nome:
    print(f"GPU:           {gpu_nome}")
    print(f"Driver NVIDIA: {driver_ver}")
else:
    # Caso não seja NVIDIA ou o driver não esteja instalado
    lspci_gpu = comando(["sh", "-c", "lspci | grep -E -i 'vga|3d|display'"])
    print(f"GPU:           {lspci_gpu if lspci_gpu else 'Nenhuma GPU detectada ou nvidia-smi indisponível'}")

# ---- GPU vista pelo PyTorch (CUDA) ----
print("\n=== PyTorch + GPU (CUDA) ===")
cuda_disponivel = torch.cuda.is_available()
print(f"CUDA compilado:  {torch.backends.cuda.is_built()}")
print(f"CUDA disponível: {cuda_disponivel}")

if cuda_disponivel:
    device = "cuda"
    gpu_id = torch.cuda.current_device()
    print(f"Dispositivo:     {torch.cuda.get_device_name(gpu_id)}")
    print(f"Versão CUDA:     {torch.version.cuda}")
    print(f"VRAM Total:      {torch.cuda.get_device_properties(gpu_id).total_memory / 1024**3:.2f} GB")

    # Teste de multiplicação de matrizes na GPU
    a = torch.rand(2000, 2000, device=device)
    b = torch.rand(2000, 2000, device=device)
    c = a @ b
    torch.cuda.synchronize()

    mem_alocada = torch.cuda.memory_allocated(device=gpu_id) / 1024**2
    mem_reservada = torch.cuda.memory_reserved(device=gpu_id) / 1024**2

    print(f"Teste na GPU:    OK (resultado em {c.device})")
    print(f"VRAM Alocada:    {mem_alocada:.1f} MB")
    print(f"VRAM Reservada:  {mem_reservada:.1f} MB")
else:
    print("PyTorch operando em modo CPU.")

=== Software ===
Python:        3.13.5 (x86_64)
PyTorch:       2.6.0+cu124
Transformers:  5.19.0
Linux:         Debian GNU/Linux 13 (trixie) (Kernel 6.12.111+deb13-amd64)

=== Hardware ===
Processador:   12th Gen Intel(R) Core(TM) i5-12400F
Núcleos CPU:   12
Memória RAM:   15.5 GB

=== GPU (Sistema) ===
GPU:           NVIDIA GeForce RTX 3060
Driver NVIDIA: 550.163.01

=== PyTorch + GPU (CUDA) ===
CUDA compilado:  True
CUDA disponível: True
Dispositivo:     NVIDIA GeForce RTX 3060
Versão CUDA:     12.4
VRAM Total:      7.78 GB
Teste na GPU:    OK (resultado em cuda:0)
VRAM Alocada:    56.1 MB
VRAM Reservada:  68.0 MB


In [2]:
"""
01 - Conhecendo o SmolLM2-135M
==============================
Carrega o modelo, mostra quantos parâmetros ele tem, como o texto vira tokens
e como o modelo gera texto (prevendo um token de cada vez).

Rodar:  python 01_modelo.py
Script independente: não importa nada de outros arquivos da pasta.
"""
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO = "HuggingFaceTB/SmolLM2-135M"   # modelo BASE (não é o Instruct)

# Escolhe o melhor hardware disponível: GPU NVIDIA, GPU Apple (Mac M1/M2/M3) ou CPU
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Hardware: {device}")

# %% 1) Carregar tokenizador e modelo (o download acontece só na 1ª vez)
tokenizer = AutoTokenizer.from_pretrained(MODELO)
model = AutoModelForCausalLM.from_pretrained(MODELO).to(device)
model.eval()

# %% 2) Anatomia: quantos parâmetros e como estão organizados
total = sum(p.numel() for p in model.parameters())
cfg = model.config
print(f"\nParâmetros: {total:,}".replace(",", "."))
print(f"Camadas (blocos transformer): {cfg.num_hidden_layers}")
print(f"Dimensão do embedding:        {cfg.hidden_size}")
print(f"Cabeças de atenção:           {cfg.num_attention_heads}")
print(f"Tamanho do vocabulário:       {cfg.vocab_size}")

# %% 3) Texto -> tokens -> texto
frase = "def somar(a, b):\n    return a + b"
ids = tokenizer(frase)["input_ids"]
print(f"\nFrase: {frase!r}")
print(f"{len(ids)} tokens: {ids}")
print("Pedaços:", [tokenizer.decode([i]) for i in ids])

# %% 4) O que o modelo "acha" que vem depois? (top-5 próximos tokens)
prompt = "def somar(a, b):\n    return"
entrada = tokenizer(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    logits = model(**entrada).logits[0, -1]          # scores do próximo token
probs = torch.softmax(logits, dim=-1)
top = torch.topk(probs, 5)
print(f"\nPrompt: {prompt!r}")
for p, i in zip(top.values, top.indices):
    print(f"  {tokenizer.decode([int(i)])!r:>12}  {p.item():.1%}")

# %% 5) Geração: repetir a previsão várias vezes
prompt = "# Função em Python que calcula o fatorial de n\ndef fatorial(n):"
entrada = tokenizer(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    saida = model.generate(**entrada, max_new_tokens=60, do_sample=False,
                           pad_token_id=tokenizer.eos_token_id)
print("\n--- Geração do modelo base ---")
print(tokenizer.decode(saida[0], skip_special_tokens=True))


Hardware: cuda


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]


Parâmetros: 134.515.008
Camadas (blocos transformer): 30
Dimensão do embedding:        576
Cabeças de atenção:           9
Tamanho do vocabulário:       49152

Frase: 'def somar(a, b):\n    return a + b'
13 tokens: [1604, 1097, 280, 24, 81, 28, 278, 727, 472, 1003, 253, 1232, 278]
Pedaços: ['def', ' som', 'ar', '(', 'a', ',', ' b', '):', '\n   ', ' return', ' a', ' +', ' b']

Prompt: 'def somar(a, b):\n    return'
          ' a'  73.4%
          ' ('  7.8%
        ' sum'  2.4%
        ' int'  2.1%
          ' b'  1.7%

--- Geração do modelo base ---
# Função em Python que calcula o fatorial de n
def fatorial(n):
    if n == 0:
        return 1
    else:
        return n * fatorial(n-1)

print(fatorial(10))



In [3]:
"""
02 - Criando o dataset de treino (tarefa -> código Python)
==========================================================
Gera exemplos no formato:

    ### Tarefa:
    Escreva uma função em Python chamada `somar_pares(numeros)` que ...
    ### Código:
    def somar_pares(numeros):
        return sum(x for x in numeros if x % 2 == 0)

Cada exemplo vem com TESTES (asserts). É isso que permite avaliar o modelo
de forma objetiva: o código gerado passa nos testes ou não passa.

Saída:
  dados/treino.jsonl  -> usado no treino
  dados/teste.jsonl   -> NUNCA visto no treino. Dois tipos:
      "variacao" = mesma família de tarefa, mas nome/enunciado diferentes
      "inedita"  = família de tarefa que o modelo nunca viu

Rodar:  python 02_dados.py
"""
import json
import random
from pathlib import Path

random.seed(42)

# Cada família: nomes possíveis da função, nomes do argumento,
# frases que descrevem o que ela faz, corpo da função e testes.
# {a} = nome do argumento, {f} = nome da função
FAMILIAS = {
    "soma_pares": dict(
        nomes=["somar_pares", "soma_pares", "total_pares"],
        args=["numeros", "lista", "valores"],
        oque=["retorna a soma dos números pares da lista",
              "soma apenas os valores pares de uma lista"],
        corpo="    return sum(x for x in {a} if x % 2 == 0)",
        testes=["{f}([1, 2, 3, 4]) == 6", "{f}([]) == 0", "{f}([5, 7]) == 0"]),
    "soma_impares": dict(
        nomes=["somar_impares", "soma_impares", "total_impares"],
        args=["numeros", "lista", "valores"],
        oque=["retorna a soma dos números ímpares da lista",
              "soma apenas os valores ímpares de uma lista"],
        corpo="    return sum(x for x in {a} if x % 2 != 0)",
        testes=["{f}([1, 2, 3, 4]) == 4", "{f}([]) == 0", "{f}([2, 8]) == 0"]),
    "maior": dict(
        nomes=["maior", "maior_valor", "maximo"],
        args=["numeros", "lista", "valores"],
        oque=["retorna o maior número da lista",
              "devolve o maior valor de uma lista"],
        corpo="    return max({a})",
        testes=["{f}([3, 9, 2]) == 9", "{f}([-5, -1]) == -1"]),
    "menor": dict(
        nomes=["menor", "menor_valor", "minimo"],
        args=["numeros", "lista", "valores"],
        oque=["retorna o menor número da lista",
              "devolve o menor valor de uma lista"],
        corpo="    return min({a})",
        testes=["{f}([3, 9, 2]) == 2", "{f}([-5, -1]) == -5"]),
    "media": dict(
        nomes=["media", "calcular_media", "valor_medio"],
        args=["numeros", "lista", "notas"],
        oque=["retorna a média aritmética dos números da lista",
              "calcula a média dos valores de uma lista"],
        corpo="    return sum({a}) / len({a})",
        testes=["{f}([2, 4]) == 3", "{f}([5]) == 5"]),
    "soma_quadrados": dict(
        nomes=["soma_quadrados", "somar_quadrados", "total_quadrados"],
        args=["numeros", "lista", "valores"],
        oque=["retorna a soma dos quadrados dos números da lista",
              "soma o quadrado de cada valor de uma lista"],
        corpo="    return sum(x * x for x in {a})",
        testes=["{f}([1, 2, 3]) == 14", "{f}([]) == 0"]),
    "contar_positivos": dict(
        nomes=["contar_positivos", "qtd_positivos", "conta_positivos"],
        args=["numeros", "lista", "valores"],
        oque=["retorna quantos números positivos existem na lista",
              "conta os valores maiores que zero de uma lista"],
        corpo="    return sum(1 for x in {a} if x > 0)",
        testes=["{f}([1, -2, 3, 0]) == 2", "{f}([]) == 0"]),
    "dobrar_lista": dict(
        nomes=["dobrar", "dobrar_lista", "duplicar_valores"],
        args=["numeros", "lista", "valores"],
        oque=["retorna uma nova lista com cada número multiplicado por 2",
              "devolve a lista com todos os valores dobrados"],
        corpo="    return [x * 2 for x in {a}]",
        testes=["{f}([1, 2, 3]) == [2, 4, 6]", "{f}([]) == []"]),
    "remover_negativos": dict(
        nomes=["remover_negativos", "sem_negativos", "filtrar_negativos"],
        args=["numeros", "lista", "valores"],
        oque=["retorna uma nova lista sem os números negativos",
              "remove os valores negativos de uma lista"],
        corpo="    return [x for x in {a} if x >= 0]",
        testes=["{f}([1, -2, 3, 0]) == [1, 3, 0]", "{f}([-1]) == []"]),
    "remover_duplicados": dict(
        nomes=["remover_duplicados", "sem_repetidos", "valores_unicos"],
        args=["itens", "lista", "valores"],
        oque=["retorna a lista sem elementos repetidos, mantendo a ordem",
              "remove os itens duplicados de uma lista preservando a ordem"],
        corpo="    return list(dict.fromkeys({a}))",
        testes=["{f}([1, 2, 1, 3, 2]) == [1, 2, 3]", "{f}([]) == []"]),
    "ultimo": dict(
        nomes=["ultimo", "ultimo_elemento", "pegar_ultimo"],
        args=["itens", "lista", "valores"],
        oque=["retorna o último elemento da lista",
              "devolve o item que está na última posição de uma lista"],
        corpo="    return {a}[-1]",
        testes=["{f}([1, 2, 3]) == 3", "{f}(['a']) == 'a'"]),
    "inverter_texto": dict(
        nomes=["inverter", "inverter_texto", "texto_invertido"],
        args=["texto", "frase", "s"],
        oque=["retorna o texto de trás para frente",
              "inverte a ordem dos caracteres de uma string"],
        corpo="    return {a}[::-1]",
        testes=["{f}('abc') == 'cba'", "{f}('') == ''"]),
    "maiusculas": dict(
        nomes=["maiusculas", "para_maiusculas", "gritar"],
        args=["texto", "frase", "s"],
        oque=["retorna o texto todo em letras maiúsculas",
              "converte uma string para maiúsculas"],
        corpo="    return {a}.upper()",
        testes=["{f}('abc') == 'ABC'", "{f}('Oi') == 'OI'"]),
    "contar_vogais": dict(
        nomes=["contar_vogais", "qtd_vogais", "conta_vogais"],
        args=["texto", "frase", "s"],
        oque=["retorna quantas vogais existem no texto",
              "conta as vogais (a, e, i, o, u) de uma string"],
        corpo='    return sum(1 for c in {a}.lower() if c in "aeiou")',
        testes=["{f}('Banana') == 3", "{f}('xyz') == 0"]),
    "contar_palavras": dict(
        nomes=["contar_palavras", "qtd_palavras", "conta_palavras"],
        args=["texto", "frase", "s"],
        oque=["retorna quantas palavras existem no texto",
              "conta as palavras de uma string separadas por espaço"],
        corpo="    return len({a}.split())",
        testes=["{f}('ola mundo bom') == 3", "{f}('') == 0"]),
    "eh_palindromo": dict(
        nomes=["eh_palindromo", "palindromo", "e_palindromo"],
        args=["texto", "frase", "s"],
        oque=["retorna True se o texto for um palíndromo, ignorando espaços e maiúsculas",
              "verifica se uma string é palíndromo, sem considerar espaços e maiúsculas"],
        corpo='    t = {a}.lower().replace(" ", "")\n    return t == t[::-1]',
        testes=["{f}('Ame a ema') == True", "{f}('abc') == False"]),
    "eh_par": dict(
        nomes=["eh_par", "par", "e_par"],
        args=["n", "numero", "x"],
        oque=["retorna True se o número for par e False caso contrário",
              "verifica se um número inteiro é par"],
        corpo="    return {a} % 2 == 0",
        testes=["{f}(4) == True", "{f}(7) == False"]),
    "quadrado": dict(
        nomes=["quadrado", "ao_quadrado", "elevar_quadrado"],
        args=["n", "numero", "x"],
        oque=["retorna o número elevado ao quadrado",
              "calcula o quadrado de um número"],
        corpo="    return {a} * {a}",
        testes=["{f}(3) == 9", "{f}(-2) == 4"]),
    "fatorial": dict(
        nomes=["fatorial", "calcular_fatorial", "fat"],
        args=["n", "numero", "x"],
        oque=["retorna o fatorial de n usando um laço for",
              "calcula o fatorial de um inteiro não negativo"],
        corpo="    resultado = 1\n    for i in range(2, {a} + 1):\n        resultado *= i\n    return resultado",
        testes=["{f}(5) == 120", "{f}(0) == 1"]),
    "celsius_fahrenheit": dict(
        nomes=["celsius_para_fahrenheit", "c_para_f", "converter_temperatura"],
        args=["celsius", "graus", "c"],
        oque=["converte uma temperatura de Celsius para Fahrenheit",
              "retorna a temperatura em Fahrenheit a partir de graus Celsius"],
        corpo="    return {a} * 9 / 5 + 32",
        testes=["{f}(100) == 212", "{f}(0) == 32"]),

    # ---- Famílias que NUNCA aparecem no treino (teste de generalização) ----
    "produto": dict(
        inedita=True,
        nomes=["produto", "multiplicar_todos"],
        args=["numeros", "lista"],
        oque=["retorna o produto (multiplicação) de todos os números da lista"],
        corpo="    resultado = 1\n    for x in {a}:\n        resultado *= x\n    return resultado",
        testes=["{f}([2, 3, 4]) == 24", "{f}([]) == 1"]),
    "triplicar_lista": dict(
        inedita=True,
        nomes=["triplicar", "triplicar_lista"],
        args=["numeros", "lista"],
        oque=["retorna uma nova lista com cada número multiplicado por 3"],
        corpo="    return [x * 3 for x in {a}]",
        testes=["{f}([1, 2]) == [3, 6]", "{f}([]) == []"]),
    "minusculas": dict(
        inedita=True,
        nomes=["minusculas", "para_minusculas"],
        args=["texto", "frase"],
        oque=["retorna o texto todo em letras minúsculas"],
        corpo="    return {a}.lower()",
        testes=["{f}('ABC') == 'abc'", "{f}('Oi') == 'oi'"]),
    "soma_positivos": dict(
        inedita=True,
        nomes=["somar_positivos", "soma_positivos"],
        args=["numeros", "lista"],
        oque=["retorna a soma dos números positivos da lista"],
        corpo="    return sum(x for x in {a} if x > 0)",
        testes=["{f}([1, -2, 3]) == 4", "{f}([]) == 0"]),
}

ENUNCIADOS = [
    "Escreva uma função em Python chamada `{f}({a})` que {oque}.",
    "Crie a função `{f}({a})`, que {oque}.",
    "Implemente em Python a função `{f}({a})`: ela {oque}.",
]


def montar_exemplo(familia, f, a, oque, modelo_enunciado):
    fam = FAMILIAS[familia]
    tarefa = modelo_enunciado.format(f=f, a=a, oque=oque)
    codigo = f"def {f}({a}):\n" + fam["corpo"].format(a=a)
    return {
        "familia": familia,
        "tarefa": tarefa,
        "codigo": codigo,
        "testes": [t.format(f=f) for t in fam["testes"]],
    }


def todas_combinacoes(familia):
    fam = FAMILIAS[familia]
    return [(f, a, o, e) for f in fam["nomes"] for a in fam["args"]
            for o in fam["oque"] for e in ENUNCIADOS]


treino, teste = [], []
for familia, fam in FAMILIAS.items():
    combos = todas_combinacoes(familia)
    random.shuffle(combos)
    if fam.get("inedita"):
        # família inteira fica só no teste
        for c in combos[:4]:
            teste.append({**montar_exemplo(familia, *c), "tipo": "inedita"})
    else:
        # 3 combinações vão para o teste, o resto (até 25) para o treino
        for c in combos[:3]:
            teste.append({**montar_exemplo(familia, *c), "tipo": "variacao"})
        for c in combos[3:28]:
            treino.append(montar_exemplo(familia, *c))

random.shuffle(treino)
Path("dados").mkdir(exist_ok=True)
for nome, dados in [("treino", treino), ("teste", teste)]:
    with open(f"dados/{nome}.jsonl", "w", encoding="utf-8") as arq:
        for ex in dados:
            arq.write(json.dumps(ex, ensure_ascii=False) + "\n")

n_var = sum(e["tipo"] == "variacao" for e in teste)
print(f"Treino: {len(treino)} exemplos")
print(f"Teste:  {len(teste)} exemplos ({n_var} variações + {len(teste) - n_var} inéditas)")
print("\nExemplo de treino:\n")
print(f"### Tarefa:\n{treino[0]['tarefa']}\n### Código:\n{treino[0]['codigo']}")
print("\nTestes:", treino[0]["testes"])


Treino: 500 exemplos
Teste:  76 exemplos (60 variações + 16 inéditas)

Exemplo de treino:

### Tarefa:
Crie a função `e_palindromo(texto)`, que verifica se uma string é palíndromo, sem considerar espaços e maiúsculas.
### Código:
def e_palindromo(texto):
    t = texto.lower().replace(" ", "")
    return t == t[::-1]

Testes: ["e_palindromo('Ame a ema') == True", "e_palindromo('abc') == False"]


In [4]:
"""
03 - Avaliando o modelo: o código gerado PASSA nos testes?
==========================================================
Para cada tarefa do conjunto de teste:
  1. o modelo gera o código a partir do enunciado;
  2. o código é executado junto com os asserts (em um processo separado);
  3. conta-se quantas tarefas passaram  ->  taxa de acerto (pass@1).

Qual modelo avaliar? Duas formas:
  a) No VSCode (células # %%): mude USAR_MODELO_TREINADO abaixo.
     False = modelo base do Hugging Face (ANTES do treino)
     True  = pasta modelo-treinado/     (DEPOIS do treino)
  b) No terminal:
     python 03_avaliar.py                    -> modelo base
     python 03_avaliar.py modelo-treinado    -> modelo treinado
"""
import json
import os
import subprocess
import sys
import time
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

USAR_MODELO_TREINADO = False          # <-- troque para True depois do 04_treinar.py

BASE = "HuggingFaceTB/SmolLM2-135M"   # SmolLM2-135M original, baixado do Hugging Face
MODELO = "modelo-treinado" if USAR_MODELO_TREINADO else BASE
# Argumento de terminal (ignorado no VSCode/Jupyter, que passa "-f ...")
argumentos = sys.argv[1:]
if argumentos and not any(a.startswith("-") for a in argumentos):
    MODELO = argumentos[0]
MAX_TOKENS = 100

# Garante que as pastas dados/ e resultados/ sejam as ao lado deste script
try:
    os.chdir(Path(__file__).resolve().parent)
except NameError:                     # janela interativa sem __file__
    pass

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

# %% Carregar modelo e dados de teste
tokenizer = AutoTokenizer.from_pretrained(MODELO)
model = AutoModelForCausalLM.from_pretrained(MODELO).to(device)
model.eval()
teste = [json.loads(l) for l in open("dados/teste.jsonl", encoding="utf-8")]
print(f"Modelo: {MODELO}  |  hardware: {device}  |  {len(teste)} tarefas\n")


def gerar_codigo(tarefa):
    """Usa exatamente o mesmo formato de prompt do treino."""
    prompt = f"### Tarefa:\n{tarefa}\n### Código:\n"
    entrada = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        saida = model.generate(**entrada, max_new_tokens=MAX_TOKENS, do_sample=False,
                               pad_token_id=tokenizer.eos_token_id)
    novos = saida[0][entrada["input_ids"].shape[1]:]
    return tokenizer.decode(novos, skip_special_tokens=True).lstrip("\n")


def extrair_funcao(texto):
    """Procura o primeiro 'def' (ignorando texto e ``` antes dele) e fica só
    com a função: para na 1ª linha não indentada depois do 'def'."""
    todas = [l for l in texto.split("\n") if not l.strip().startswith("```")]
    inicio = next((i for i, l in enumerate(todas) if l.startswith("def ")), None)
    if inicio is None:
        return texto.strip()          # não escreveu função nenhuma
    linhas = [todas[inicio]]
    for linha in todas[inicio + 1:]:
        if linha.strip() and not linha.startswith((" ", "\t")):
            break
        linhas.append(linha)
    return "\n".join(linhas).rstrip()


def resumo(codigo, n=45):
    """Primeira linha do que o modelo escreveu, para mostrar na tela."""
    primeira = codigo.strip().split("\n")[0] if codigo.strip() else "(vazio)"
    return primeira[:n] + ("..." if len(primeira) > n else "")


def passa_nos_testes(codigo, testes):
    """Executa código + asserts num processo separado (com tempo limite)."""
    programa = codigo + "\n\n" + "\n".join(f"assert {t}" for t in testes) + "\n"
    programa = programa.replace("\x00", "")          # lixo que o modelo às vezes gera
    try:
        r = subprocess.run([sys.executable, "-c", programa],
                           capture_output=True, text=True, timeout=5)
    except subprocess.TimeoutExpired:
        return False, "tempo esgotado (loop infinito?)"
    except Exception as e:
        return False, f"não executou: {e}"
    if r.returncode == 0:
        return True, ""
    erro = r.stderr.strip().splitlines()
    return False, erro[-1] if erro else "erro"


# %% Avaliar todas as tarefas
resultados = []
inicio = time.time()
for i, ex in enumerate(teste, 1):
    codigo = extrair_funcao(gerar_codigo(ex["tarefa"]))
    ok, erro = passa_nos_testes(codigo, ex["testes"])
    resultados.append({**ex, "gerado": codigo, "passou": ok, "erro": erro})
    print(f"[{i:2d}/{len(teste)}] {'PASSOU' if ok else 'falhou'}  {ex['familia']:<20} "
          f"gerou: {resumo(codigo)!r}  {erro}")

# %% Resumo
print(f"\nTempo: {time.time() - inicio:.0f} s")
print("=" * 50)
for tipo in ["variacao", "inedita"]:
    grupo = [r for r in resultados if r["tipo"] == tipo]
    acertos = sum(r["passou"] for r in grupo)
    print(f"{tipo:<10}: {acertos:2d}/{len(grupo)}  ({acertos / len(grupo):.0%})")
total = sum(r["passou"] for r in resultados)
print(f"{'TOTAL':<10}: {total:2d}/{len(resultados)}  ({total / len(resultados):.0%})")
print("=" * 50)

# Mostra 3 exemplos completos lado a lado com a resposta esperada
for r in resultados[:3]:
    print(f"\nTAREFA: {r['tarefa']}")
    print(f"--- esperado ---\n{r['codigo']}")
    print(f"--- gerado ({'passou' if r['passou'] else 'falhou'}) ---\n{r['gerado']}")

# %% Salvar para comparar depois
Path("resultados").mkdir(exist_ok=True)
nome = "base" if MODELO == BASE else Path(MODELO).name
with open(f"resultados/{nome}.json", "w", encoding="utf-8") as arq:
    json.dump(resultados, arq, ensure_ascii=False, indent=2)
print(f"\nResultados salvos em resultados/{nome}.json")


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Modelo: HuggingFaceTB/SmolLM2-135M  |  hardware: cuda  |  76 tarefas

[ 1/76] falhou  soma_pares           gerou: 'def total_pares(numeros):'  AssertionError
[ 2/76] PASSOU  soma_pares           gerou: 'def soma_pares(numeros):'  
[ 3/76] falhou  soma_pares           gerou: 'def total_pares(lista):'  AssertionError
[ 4/76] falhou  soma_impares         gerou: 'def total_impares(lista):'  AssertionError
[ 5/76] falhou  soma_impares         gerou: 'def somar_impares(lista):'  AssertionError
[ 6/76] falhou  soma_impares         gerou: 'def somar_impares(numeros):'  AssertionError
[ 7/76] PASSOU  maior                gerou: 'def maior(lista):'  
[ 8/76] PASSOU  maior                gerou: 'def maior(lista):'  
[ 9/76] PASSOU  maior                gerou: 'def maximo(lista):'  
[10/76] PASSOU  menor                gerou: 'def minimo(numeros):'  
[11/76] PASSOU  menor                gerou: 'def menor(valores):'  
[12/76] PASSOU  menor                gerou: 'def menor_valor(lista):'  
[13/76] P

In [5]:
"""
04 - Treinando (fine-tuning) o SmolLM2-135M
===========================================
Laço de treino "na mão", em PyTorch puro, para enxergar cada etapa:

    para cada lote:
        1. forward  -> o modelo prevê o próximo token
        2. loss     -> o quão errada foi a previsão (cross-entropy)
        3. backward -> calcula os gradientes
        4. step     -> ajusta os 134 milhões de pesos um pouquinho

Ao final: gráfico da loss (curva_loss.png) e modelo salvo em modelo-treinado/

Rodar:  python 04_treinar.py
"""
import json
import random
import time

import matplotlib
matplotlib.use("Agg")                      # salva o gráfico sem abrir janela
import matplotlib.pyplot as plt
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from pathlib import Path

# Garante que dados/, resultados/ e modelo-treinado/ fiquem ao lado deste script
import os
try:
    os.chdir(Path(__file__).resolve().parent)
except NameError:                     # janela interativa sem __file__
    pass

MODELO = "HuggingFaceTB/SmolLM2-135M"
SAIDA = "modelo-treinado"
EPOCAS = 3
LOTE = 8                # exemplos por passo
LR = 5e-5               # taxa de aprendizado
MAX_LEN = 160           # tokens por exemplo

random.seed(0)
torch.manual_seed(0)
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Hardware: {device}")

# %% 1) Modelo, tokenizador e dados
tokenizer = AutoTokenizer.from_pretrained(MODELO)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODELO).to(device)

treino = [json.loads(l) for l in open("dados/treino.jsonl", encoding="utf-8")]
teste = [json.loads(l) for l in open("dados/teste.jsonl", encoding="utf-8")]


def preparar_lote(exemplos):
    """Transforma exemplos em tensores. O modelo só é cobrado (labels) pelo
    CÓDIGO: os tokens do enunciado e do padding recebem -100 (ignorados)."""
    ids_lote, labels_lote = [], []
    for ex in exemplos:
        prompt = f"### Tarefa:\n{ex['tarefa']}\n### Código:\n"
        resposta = ex["codigo"] + tokenizer.eos_token   # EOS ensina a PARAR
        ids_p = tokenizer(prompt)["input_ids"]
        ids_r = tokenizer(resposta, add_special_tokens=False)["input_ids"]
        ids = (ids_p + ids_r)[:MAX_LEN]
        labels = ([-100] * len(ids_p) + ids_r)[:MAX_LEN]
        ids_lote.append(ids)
        labels_lote.append(labels)

    tam = max(len(x) for x in ids_lote)
    input_ids = [x + [tokenizer.pad_token_id] * (tam - len(x)) for x in ids_lote]
    mascara = [[1] * len(x) + [0] * (tam - len(x)) for x in ids_lote]
    labels = [x + [-100] * (tam - len(x)) for x in labels_lote]
    return {
        "input_ids": torch.tensor(input_ids, device=device),
        "attention_mask": torch.tensor(mascara, device=device),
        "labels": torch.tensor(labels, device=device),
    }


def loss_no_teste():
    """Loss média em dados que o modelo NÃO usa para aprender."""
    model.eval()
    total = 0.0
    with torch.no_grad():
        for i in range(0, len(teste), LOTE):
            total += model(**preparar_lote(teste[i:i + LOTE])).loss.item()
    model.train()
    return total / ((len(teste) + LOTE - 1) // LOTE)


# %% 2) Laço de treino
otimizador = torch.optim.AdamW(model.parameters(), lr=LR)
passos_por_epoca = (len(treino) + LOTE - 1) // LOTE
print(f"{len(treino)} exemplos | {passos_por_epoca} passos por época | {EPOCAS} épocas\n")

hist_passo, hist_loss = [], []
hist_val_passo, hist_val = [0], [loss_no_teste()]
print(f"Loss no teste ANTES do treino: {hist_val[0]:.3f}")

model.train()
passo = 0
inicio = time.time()
for epoca in range(1, EPOCAS + 1):
    random.shuffle(treino)
    for i in range(0, len(treino), LOTE):
        lote = preparar_lote(treino[i:i + LOTE])
        loss = model(**lote).loss        # 1 + 2) forward e loss
        loss.backward()                  # 3) gradientes
        otimizador.step()                # 4) atualiza os pesos
        otimizador.zero_grad()

        passo += 1
        hist_passo.append(passo)
        hist_loss.append(loss.item())
        if passo % 10 == 0:
            print(f"época {epoca} | passo {passo:4d} | loss {loss.item():.3f} "
                  f"| {time.time() - inicio:5.0f} s")

    val = loss_no_teste()
    hist_val_passo.append(passo)
    hist_val.append(val)
    print(f"--> fim da época {epoca}: loss no teste = {val:.3f}\n")

print(f"Treino concluído em {time.time() - inicio:.0f} s")

# %% 3) Gráfico da curva de aprendizado
plt.figure(figsize=(8, 4.5))
plt.plot(hist_passo, hist_loss, alpha=0.5, label="treino (cada passo)")
plt.plot(hist_val_passo, hist_val, "o-", linewidth=2, label="teste (fim de cada época)")
plt.xlabel("passo de treino")
plt.ylabel("loss (cross-entropy)")
plt.title("Fine-tuning do SmolLM2-135M: tarefa -> código Python")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig("curva_loss.png", dpi=120)
print("Gráfico salvo em curva_loss.png")

# %% 4) Salvar o modelo treinado
model.save_pretrained(SAIDA)
tokenizer.save_pretrained(SAIDA)
print(f"Modelo salvo em {SAIDA}/")
print("Próximo passo: no 03_avaliar.py troque USAR_MODELO_TREINADO para True e rode de novo")


Hardware: cuda


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

500 exemplos | 63 passos por época | 3 épocas

Loss no teste ANTES do treino: 0.802
época 1 | passo   10 | loss 0.350 |     1 s
época 1 | passo   20 | loss 0.394 |     2 s
época 1 | passo   30 | loss 0.136 |     3 s
época 1 | passo   40 | loss 0.133 |     4 s
época 1 | passo   50 | loss 0.215 |     6 s
época 1 | passo   60 | loss 0.099 |     7 s
--> fim da época 1: loss no teste = 0.119

época 2 | passo   70 | loss 0.150 |     8 s
época 2 | passo   80 | loss 0.105 |     9 s
época 2 | passo   90 | loss 0.075 |    10 s
época 2 | passo  100 | loss 0.049 |    12 s
época 2 | passo  110 | loss 0.068 |    13 s
época 2 | passo  120 | loss 0.040 |    14 s
--> fim da época 2: loss no teste = 0.065

época 3 | passo  130 | loss 0.040 |    15 s
época 3 | passo  140 | loss 0.044 |    16 s
época 3 | passo  150 | loss 0.056 |    18 s
época 3 | passo  160 | loss 0.037 |    19 s
época 3 | passo  170 | loss 0.019 |    20 s
época 3 | passo  180 | loss 0.020 |    21 s
--> fim da época 3: loss no teste = 0.

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo salvo em modelo-treinado/
Próximo passo: no 03_avaliar.py troque USAR_MODELO_TREINADO para True e rode de novo


In [6]:
"""
05 - Antes x depois: modelo base contra modelo treinado
=======================================================
1. Mostra o placar salvo pelo 03_avaliar.py (base e treinado).
2. Faz a MESMA pergunta aos dois modelos, lado a lado.
   Troque o texto de PERGUNTAS para testar ao vivo com a turma.

Rodar:  python 05_comparar.py
"""
import json
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Garante que dados/, resultados/ e modelo-treinado/ fiquem ao lado deste script
import os
try:
    os.chdir(Path(__file__).resolve().parent)
except NameError:                     # janela interativa sem __file__
    pass

BASE = "HuggingFaceTB/SmolLM2-135M"
TREINADO = "modelo-treinado"
PERGUNTAS = [
    "Escreva uma função em Python chamada `somar_pares(valores)` que retorna a soma dos números pares da lista.",
    "Crie a função `contar_letras_a(texto)`, que retorna quantas letras 'a' existem no texto.",
]

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

# %% 1) Placar
print("PLACAR (taxa de código que passa nos testes)")
print(f"{'modelo':<18}{'variação':>10}{'inédita':>10}{'total':>10}")
for nome in ["base", TREINADO]:
    arq = Path(f"resultados/{nome}.json")
    if not arq.exists():
        print(f"{nome:<18}  (rode antes: python 03_avaliar.py"
              f"{'' if nome == 'base' else ' ' + TREINADO})")
        continue
    res = json.loads(arq.read_text(encoding="utf-8"))
    taxa = lambda g: f"{sum(r['passou'] for r in g) / len(g):.0%}"
    print(f"{nome:<18}{taxa([r for r in res if r['tipo'] == 'variacao']):>10}"
          f"{taxa([r for r in res if r['tipo'] == 'inedita']):>10}{taxa(res):>10}")


# %% 2) Mesma pergunta para os dois modelos
def responder(model, tokenizer, tarefa):
    prompt = f"### Tarefa:\n{tarefa}\n### Código:\n"
    entrada = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        saida = model.generate(**entrada, max_new_tokens=80, do_sample=False,
                               pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(saida[0][entrada["input_ids"].shape[1]:],
                            skip_special_tokens=True).strip()


modelos = {}
for nome, caminho in [("BASE", BASE), ("TREINADO", TREINADO)]:
    tok = AutoTokenizer.from_pretrained(caminho)
    mod = AutoModelForCausalLM.from_pretrained(caminho).to(device).eval()
    modelos[nome] = (mod, tok)

for tarefa in PERGUNTAS:
    print("\n" + "=" * 70)
    print(f"TAREFA: {tarefa}")
    for nome, (mod, tok) in modelos.items():
        print(f"\n--- {nome} ---")
        print(responder(mod, tok, tarefa))


PLACAR (taxa de código que passa nos testes)
modelo              variação   inédita     total
base                     42%        0%       33%
modelo-treinado     (rode antes: python 03_avaliar.py modelo-treinado)


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]


TAREFA: Escreva uma função em Python chamada `somar_pares(valores)` que retorna a soma dos números pares da lista.

--- BASE ---
```python
def somar_pares(valores):
    return sum(valores)
```
### Como executar:
```python
soma_pares([1, 2, 3, 4, 5])
```
### Como usar:
```python
soma_pares([1, 2, 3,

--- TREINADO ---
def somar_pares(valores):
    return sum(x for x in valores if x % 2 == 0)

TAREFA: Crie a função `contar_letras_a(texto)`, que retorna quantas letras 'a' existem no texto.

--- BASE ---
```python
def contar_letras_a(texto):
    letras = texto.split()
    for letra in letras:
        if letra.isalpha():
            return int(letra)
    return 0
```
### Código:
```python
def contar_letras_a(texto):
    letras =

--- TREINADO ---
def contar_letras_a(texto):
    return len(texto.split())
